OUTAGE DETECTED: getting weather data when you know the location

In [6]:
import requests
from requests.exceptions import HTTPError, Timeout, RequestException

def make_nws_request(endpoint, user_agent):
    headers = {
        "User-Agent": user_agent,
    }

    try:
        response = requests.get(
                       endpoint, 
                       headers=headers
                   )
        # Raise HTTPError for bad responses (4xx or 5xx)
        response.raise_for_status()
        return response.json()

    except HTTPError as http_err:
        print(f"HTTP error occurred: {http_err} - Status code: {response.status_code}")
    except Timeout as timeout_err:
        print(f"Request timed out: {timeout_err}")
    except RequestException as req_err:
        print(f"Request error: {req_err}")
    
    return None  # Return None if an error occurred


if __name__ == "__main__":
    # Sample user agent
    user_agent = "MyWeatherApp/1.0"

    BASE_URL = "https://api.weather.gov"
    
    # Example 1: Get forecast for a specific location
    lat, lon = 39.7456, -9.0892
    forecast_url = f"{BASE_URL}/points/{lat},{lon}"
    data = make_nws_request(forecast_url, user_agent)
    
    if data:
        print(data)
    else:
        print("Failed to retrieve data.")

{'@context': ['https://geojson.org/geojson-ld/geojson-context.jsonld', {'@version': '1.1', 'wx': 'https://api.weather.gov/ontology#', 's': 'https://schema.org/', 'geo': 'http://www.opengis.net/ont/geosparql#', 'unit': 'http://codes.wmo.int/common/unit/', '@vocab': 'https://api.weather.gov/ontology#', 'geometry': {'@id': 's:GeoCoordinates', '@type': 'geo:wktLiteral'}, 'city': 's:addressLocality', 'state': 's:addressRegion', 'distance': {'@id': 's:Distance', '@type': 's:QuantitativeValue'}, 'bearing': {'@type': 's:QuantitativeValue'}, 'value': {'@id': 's:value'}, 'unitCode': {'@id': 's:unitCode', '@type': '@id'}, 'forecastOffice': {'@type': '@id'}, 'forecastGridData': {'@type': '@id'}, 'publicZone': {'@type': '@id'}, 'county': {'@type': '@id'}}], 'id': 'https://api.weather.gov/points/39.7456,-90.0892', 'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-90.0892, 39.7456]}, 'properties': {'@id': 'https://api.weather.gov/points/39.7456,-90.0892', '@type': 'wx:Point', 'cwa': '

In [8]:
import requests
# Define the API endpoint and parameters
endpoint = "https://api.weather.gov/gridpoints/ILX/32,53/forecast"
headers = {
   "User-Agent": "MyWeatherApp (myemail@example.com)",
   "Accept": "application/geo+json"
}
# Make the request to the API
response = requests.get(endpoint, headers=headers)
# Check if the request was successful
if response.status_code == 200:
   # Parse the JSON response
   data = response.json()
   # Extract the current weather conditions
   current_conditions = data["properties"]["periods"][0]
   print(f"Current temperature: {current_conditions['temperature']} {current_conditions['temperatureUnit']}")
   print(f"Forecast: {current_conditions['shortForecast']}")
else:
   print(f"Error: {response.status_code}")

Current temperature: 61 F
Forecast: Partly Cloudy then Slight Chance Rain Showers


In [9]:
print(f"Current temperature: {current_conditions}")

Current temperature: {'number': 1, 'name': 'Tonight', 'startTime': '2025-09-19T19:00:00-05:00', 'endTime': '2025-09-20T06:00:00-05:00', 'isDaytime': False, 'temperature': 61, 'temperatureUnit': 'F', 'temperatureTrend': '', 'probabilityOfPrecipitation': {'unitCode': 'wmoUnit:percent', 'value': 16}, 'windSpeed': '3 mph', 'windDirection': 'SW', 'icon': 'https://api.weather.gov/icons/land/night/sct/rain_showers,20?size=medium', 'shortForecast': 'Partly Cloudy then Slight Chance Rain Showers', 'detailedForecast': 'A slight chance of rain showers between midnight and 4am. Partly cloudy, with a low around 61. Southwest wind around 3 mph. Chance of precipitation is 20%.'}


WEATHER MONITORING: returning weather information and location
* second block is more useful but 3rd block has best response it just takes a while to load cuz it filters out test messages

In [16]:
import requests

headers = {'User-Agent' : 'myapp'}
endpoint = 'https://api.weather.gov/alerts/active'

response = requests.get(endpoint, headers = headers)
data = response.json()

# `features` contains the data we want.
print(data) 

{'@context': ['https://geojson.org/geojson-ld/geojson-context.jsonld', {'@version': '1.1', 'wx': 'https://api.weather.gov/ontology#', '@vocab': 'https://api.weather.gov/ontology#'}], 'type': 'FeatureCollection', 'features': [{'id': 'https://api.weather.gov/alerts/urn:oid:2.49.0.1.840.0-KEEPALIVE-21279', 'type': 'Feature', 'geometry': None, 'properties': {'@id': 'https://api.weather.gov/alerts/urn:oid:2.49.0.1.840.0-KEEPALIVE-21279', '@type': 'wx:Alert', 'id': 'urn:oid:2.49.0.1.840.0-KEEPALIVE-21279', 'areaDesc': 'Montgomery', 'geocode': {'SAME': ['024031'], 'UGC': ['MDC031']}, 'affectedZones': ['https://api.weather.gov/zones/county/MDC031'], 'references': [], 'sent': '2025-09-20T14:41:49+00:00', 'effective': '2025-09-20T14:41:49+00:00', 'onset': None, 'expires': '2025-09-20T14:51:49+00:00', 'ends': None, 'status': 'Test', 'messageType': 'Alert', 'category': 'Met', 'severity': 'Unknown', 'certainty': 'Unknown', 'urgency': 'Unknown', 'event': 'Test Message', 'sender': 'w-nws.webmaster@no

In [2]:
import requests

# Fetch active severe weather alerts
headers = {'User-Agent' : 'myapp'}
endpoint = 'https://api.weather.gov/alerts/active'

response = requests.get(endpoint, headers = headers)
data = response.json()

# Extract relevant data from alerts
alerts_info = []
for feature in data.get("features", []):
    properties = feature.get("properties", {})
    geometry = feature.get("geometry")

    alert_id = feature.get("id")
    event = properties.get("event")
    description = properties.get("description", "")
    area_desc = properties.get("areaDesc", "")
    effective = properties.get("effective")
    expires = properties.get("expires")
    severity = properties.get("severity")
    certainty = properties.get("certainty")
    urgency = properties.get("urgency")

    # Extract coordinates if available
    region_coords = None
    if geometry and "coordinates" in geometry:
        region_coords = geometry["coordinates"]

    alert_info = {
        "id": alert_id,
        "event": event,
        "reason": description[:300],  # Trim long text
        "area": area_desc,
        "severity": severity,
        "urgency": urgency,
        "certainty": certainty,
        "effective": effective,
        "expires": expires,
        "region_coords": region_coords
    }
    alerts_info.append(alert_info)

# Print the first 3 alerts for preview
for alert in alerts_info[:3]:
    print(f"Event: {alert['event']}")
    print(f"Area: {alert['area']}")
    print(f"Reason: {alert['reason']}")
    print(f"Coords: {alert['region_coords'][:1] if alert['region_coords'] else 'N/A'}")
    print("-" * 40)


Event: Special Weather Statement
Area: Southern Johnson County; Southern Pope County; Western and Northern Logan County; Johnson County Higher Elevations; Pope County Higher Elevations
Reason: At 944 AM CDT, Doppler radar was tracking a strong thunderstorm near
Woodland, or 10 miles northwest of Clarksville, moving southeast at
25 mph.

HAZARD...Winds in excess of 40 mph and pea size hail.

SOURCE...Radar indicated.

IMPACT...Gusty winds could knock down tree limbs and blow around
unsecur
Coords: [[[-93.7, 35.63], [-93.51, 35.7299999], [-93.08, 35.7299999], [-93.2, 35.4099999], [-93.48, 35.28], [-93.51, 35.28], [-93.7099999, 35.42], [-93.7, 35.63]]]
----------------------------------------
Event: Test Message
Area: Montgomery
Reason: Monitoring message only. Please disregard.
Coords: N/A
----------------------------------------
Event: Small Craft Advisory
Area: Whitefish Bay (U.S. Portion)/Whitefish Point to Point Iroquois MI; St. Marys River Point Iroquois to E. Potagannissing Bay
Rea

In [5]:
import requests

# --- Config ---
API_BASE = "https://api.weather.gov"
# HEADERS = {
#     "User-Agent": "MyWeatherAlertSystem (you@example.com)"  # Replace with your info
# }

# # --- Step 1: Fetch active severe alerts ---
# params = {
#     "severity": "severe",
#     "urgency": "immediate"
# }
# response = requests.get(f"{API_BASE}/alerts/active", params=params, headers=HEADERS)
# data = response.json()

import requests

# Fetch active severe weather alerts
HEADERS = {'User-Agent' : 'myapp'}
endpoint = 'https://api.weather.gov/alerts/active'

response = requests.get(endpoint, headers = HEADERS)
data = response.json()

alerts_info = []

for feature in data.get("features", []):
    properties = feature.get("properties", {})
    geometry = feature.get("geometry")
    affected_zones = properties.get("affectedZones", [])

    event = properties.get("event")
    if not event or event.lower() == "test message":
        continue  # Skip test messages

    # Extract base info
    alert_info = {
        "id": feature.get("id"),
        "event": event,
        "reason": properties.get("description", "")[:300],
        "area": properties.get("areaDesc", ""),
        "severity": properties.get("severity"),
        "urgency": properties.get("urgency"),
        "certainty": properties.get("certainty"),
        "effective": properties.get("effective"),
        "expires": properties.get("expires"),
        "region_coords": None
    }

    # --- Step 2: Use primary geometry if available ---
    if geometry and "coordinates" in geometry:
        alert_info["region_coords"] = geometry["coordinates"]
    else:
        # --- Step 3: Fallback: Try fetching geometry for first affected zone ---
        if affected_zones:
            zone_id = affected_zones[0].split("/")[-1]  # Extract zone ID
            zone_type = affected_zones[0].split("/")[-2]  # e.g. "forecast", "county"
            zone_url = f"{API_BASE}/zones/{zone_type}/{zone_id}"

            zone_response = requests.get(zone_url, headers=HEADERS)
            if zone_response.status_code == 200:
                zone_data = zone_response.json()
                zone_geom = zone_data.get("geometry")
                if zone_geom and "coordinates" in zone_geom:
                    alert_info["region_coords"] = zone_geom["coordinates"]

    alerts_info.append(alert_info)

# --- Step 4: Display summary ---
for alert in alerts_info[:5]:  # limit for brevity
    print(f"Event: {alert['event']}")
    print(f"Area: {alert['area']}")
    print(f"Reason: {alert['reason']}")
    if alert['region_coords']:
        print(f"Coords: {alert['region_coords'][:1]}...")  # preview only
    else:
        print("Coords: N/A")
    print("-" * 40)


Event: Special Weather Statement
Area: Southern Johnson County; Southern Pope County; Western and Northern Logan County; Johnson County Higher Elevations; Pope County Higher Elevations
Reason: At 944 AM CDT, Doppler radar was tracking a strong thunderstorm near
Woodland, or 10 miles northwest of Clarksville, moving southeast at
25 mph.

HAZARD...Winds in excess of 40 mph and pea size hail.

SOURCE...Radar indicated.

IMPACT...Gusty winds could knock down tree limbs and blow around
unsecur
Coords: [[[-93.7, 35.63], [-93.51, 35.7299999], [-93.08, 35.7299999], [-93.2, 35.4099999], [-93.48, 35.28], [-93.51, 35.28], [-93.7099999, 35.42], [-93.7, 35.63]]]...
----------------------------------------
Event: Small Craft Advisory
Area: Whitefish Bay (U.S. Portion)/Whitefish Point to Point Iroquois MI; St. Marys River Point Iroquois to E. Potagannissing Bay
Reason: * WHAT...Highest gusts up to 25 kts from the southeast and
highest waves around 4 feet expected.

* WHERE...Whitefish Bay (U. S. Port